# 04 — Feature Engineering

This notebook creates physically interpretable snapshot and past-only temporal features for current drivetrain-fault detection. It reuses Step 03's grouped test and temporal validation partitions, keeps the test set out of feature selection, and exports unscaled engineered data so preprocessing can remain inside later model pipelines.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
project_root = Path.cwd()
if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent
raw_path = project_root / "data" / "raw" / "wind_turbine_detection.csv"
processed_dir = project_root / "data" / "processed"

## 1. Load Data and Reuse Step 03 Partitions

In [2]:
df = pd.read_csv(raw_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")
df = df.sort_values(["turbine_id", "timestamp"]).reset_index(names="source_index")
metadata = {name: pd.read_csv(processed_dir / f"03_{name}_metadata.csv", parse_dates=["timestamp"])
            for name in ["train", "validation", "test"]}
test_ids = set(metadata["test"]["turbine_id"])
if any(test_ids & set(metadata[name]["turbine_id"]) for name in ["train", "validation"]):
    raise RuntimeError("Obsolete split metadata: rerun notebook 03 before feature engineering.")
split_lookup = pd.concat([part[["source_index"]].assign(split=name) for name, part in metadata.items()], ignore_index=True)
if split_lookup["source_index"].duplicated().any() or set(split_lookup["source_index"]) != set(df["source_index"]):
    raise RuntimeError("Step 03 metadata does not partition every source row exactly once.")
df = df.merge(split_lookup, on="source_index", how="left", validate="one_to_one")
split_report = df.groupby("split", observed=True).agg(rows=("failure", "size"), turbines=("turbine_id", "nunique"), failure_count=("failure", "sum"), start=("timestamp", "min"), end=("timestamp", "max"))
split_report["failure_pct"] = split_report["failure_count"] / split_report["rows"] * 100
display(split_report.loc[["train", "validation", "test"]])
for name, part in metadata.items():
    assert df.loc[df["split"].eq(name), "source_index"].isin(part["source_index"]).all()


,rows,turbines,failure_count,start,end,failure_pct
split,,,,,,
train,84336,12,1877,2024-01-01 00:00:00,2024-02-18 21:00:00,2.2256
validation,21072,12,1485,2024-02-18 19:10:00,2024-03-01 23:50:00,7.0473
test,26352,3,591,2024-01-01 00:00:00,2024-03-01 23:50:00,2.2427


## 2. Domain-Informed Snapshot Features

Ratios use safe denominators. `generator_rotor_ratio` and other speed-normalized features are missing when rotor speed is at or below 0.1 rpm; the separate stopped-rotor flag preserves that operating state. Angle features use circular encoding. The operating-state thresholds are transparent analytical definitions, not manufacturer specifications.

In [3]:
def safe_ratio(numerator, denominator, minimum=1e-9):
    valid = denominator.abs() > minimum
    return numerator.div(denominator.where(valid))

engineered = df.copy()
rotor_running = engineered["rotor_speed_rpm"] > 0.1
engineered["capacity_factor"] = safe_ratio(engineered["power_output_kW"], engineered["rated_power_kW"])
engineered["generator_rotor_ratio"] = safe_ratio(engineered["generator_speed_rpm"], engineered["rotor_speed_rpm"].where(rotor_running))
engineered["rotor_stopped"] = (~rotor_running).astype("int8")
engineered["gearbox_temp_rise"] = engineered["gearbox_bearing_temp_C"] - engineered["ambient_temp_C"]
engineered["main_bearing_temp_rise"] = engineered["main_bearing_temp_C"] - engineered["ambient_temp_C"]
engineered["generator_winding_temp_rise"] = engineered["generator_winding_temp_C"] - engineered["ambient_temp_C"]
engineered["gearbox_nacelle_temp_gap"] = engineered["gearbox_bearing_temp_C"] - engineered["nacelle_temp_C"]
engineered["winding_bearing_temp_gap"] = engineered["generator_winding_temp_C"] - engineered["generator_bearing_temp_C"]
fft_columns = ["vib_fft_bearing_bpfo", "vib_fft_bearing_bpfi", "vib_fft_gearmesh", "vib_fft_sideband"]
engineered["fft_mean"] = engineered[fft_columns].mean(axis=1)
engineered["fft_max"] = engineered[fft_columns].max(axis=1)
engineered["fft_spread"] = engineered[fft_columns].max(axis=1) - engineered[fft_columns].min(axis=1)
engineered["bearing_fft_sum"] = engineered["vib_fft_bearing_bpfo"] + engineered["vib_fft_bearing_bpfi"]
engineered["gearmesh_sideband_ratio"] = safe_ratio(engineered["vib_fft_gearmesh"], engineered["vib_fft_sideband"])
engineered["vibration_per_rotor_rpm"] = safe_ratio(engineered["drivetrain_vibration_rms_mmps"], engineered["rotor_speed_rpm"].where(rotor_running))
engineered["oil_particles_per_pressure"] = safe_ratio(engineered["oil_particle_count"], engineered["oil_pressure_bar"])
wind_radians = np.deg2rad(engineered["wind_direction_deg"] % 360)
yaw_radians = np.deg2rad(engineered["yaw_misalignment_deg"] % 360)
engineered["wind_direction_sin"] = np.sin(wind_radians)
engineered["wind_direction_cos"] = np.cos(wind_radians)
engineered["yaw_misalignment_sin"] = np.sin(yaw_radians)
engineered["yaw_misalignment_cos"] = np.cos(yaw_radians)
engineered["absolute_yaw_misalignment"] = engineered["yaw_misalignment_deg"].abs()
engineered["below_cutin"] = engineered["wind_speed_mps"].lt(3).astype("int8")
engineered["positive_power"] = engineered["power_output_kW"].gt(1).astype("int8")
engineered["near_rated_operation"] = engineered["capacity_factor"].ge(0.90).astype("int8")
engineered["high_pitch_state"] = engineered["blade_pitch_angle_deg"].ge(45).astype("int8")
snapshot_features = [c for c in engineered.columns if c not in df.columns]
print(f"Snapshot features created: {len(snapshot_features)}")
print(snapshot_features)

Snapshot features created: 24
['capacity_factor', 'generator_rotor_ratio', 'rotor_stopped', 'gearbox_temp_rise', 'main_bearing_temp_rise', 'generator_winding_temp_rise', 'gearbox_nacelle_temp_gap', 'winding_bearing_temp_gap', 'fft_mean', 'fft_max', 'fft_spread', 'bearing_fft_sum', 'gearmesh_sideband_ratio', 'vibration_per_rotor_rpm', 'oil_particles_per_pressure', 'wind_direction_sin', 'wind_direction_cos', 'yaw_misalignment_sin', 'yaw_misalignment_cos', 'absolute_yaw_misalignment', 'below_cutin', 'positive_power', 'near_rated_operation', 'high_pitch_state']


## 3. Past-Only Temporal Features

For six condition signals, lag, change, and rolling statistics are calculated independently by turbine. Rolling baselines are shifted by one record, so the current and future values cannot enter their own historical features. Six and eighteen records correspond to one and three hours at the verified 10-minute cadence.

In [4]:
temporal_signals = ["gearbox_bearing_temp_C", "main_bearing_temp_C", "generator_winding_temp_C", "drivetrain_vibration_rms_mmps", "oil_particle_count", "oil_pressure_bar"]
engineered = engineered.sort_values(["turbine_id", "timestamp"]).copy()
temporal_features = []
for column in temporal_signals:
    grouped = engineered.groupby("turbine_id", sort=False)[column]
    lag1_name = f"{column}_lag1"
    delta_name = f"{column}_delta1"
    engineered[lag1_name] = grouped.shift(1)
    engineered[delta_name] = engineered[column] - engineered[lag1_name]
    temporal_features.extend([lag1_name, delta_name])
    for window, label in [(6, "1h"), (18, "3h")]:
        mean_name = f"{column}_past_{label}_mean"
        std_name = f"{column}_past_{label}_std"
        shifted = grouped.shift(1)
        engineered[mean_name] = shifted.groupby(engineered["turbine_id"], sort=False).transform(lambda values: values.rolling(window, min_periods=1).mean())
        engineered[std_name] = shifted.groupby(engineered["turbine_id"], sort=False).transform(lambda values: values.rolling(window, min_periods=2).std())
        temporal_features.extend([mean_name, std_name])
print(f"Past-only temporal features created: {len(temporal_features)}")

Past-only temporal features created: 36


## 4. Feature Validation

In [5]:
new_features = snapshot_features + temporal_features
engineered[new_features] = engineered[new_features].replace([np.inf, -np.inf], np.nan)
validation_rows = []
for name in ["train", "validation", "test"]:
    part = engineered.loc[engineered["split"] == name]
    validation_rows.append({"split": name, "rows": len(part), "turbines": part["turbine_id"].nunique(), "new_features": len(new_features), "infinite_values": int(np.isinf(part[new_features].select_dtypes(include="number")).sum().sum()), "all_missing_features": int(part[new_features].isna().all().sum())})
feature_validation = pd.DataFrame(validation_rows).set_index("split")
display(feature_validation)
first_rows = engineered.groupby("turbine_id", sort=False).head(1)
lag_guard_passed = bool(first_rows[[f"{c}_lag1" for c in temporal_signals]].isna().all().all())
print("First-record lag guard passed:", lag_guard_passed)
print("Duplicate turbine/timestamp keys:", engineered.duplicated(["turbine_id", "timestamp"]).sum())

,rows,turbines,new_features,infinite_values,all_missing_features
split,,,,,
train,84336,12,60,0,0
validation,21072,12,60,0,0
test,26352,3,60,0,0


First-record lag guard passed: True
Duplicate turbine/timestamp keys: 0


## 5. Training/Validation Feature-Group Ablation

A fixed Random Forest is used only as a diagnostic to compare feature groups. This is not one of the formal baseline models. All imputers and models are fitted on training data only, selection uses validation data only, and the test set is not scored.

In [6]:
excluded = {"source_index", "timestamp", "turbine_id", "failure", "split", "prior_fault_count", "hours_since_last_maintenance", "component_age_days", "operating_hours_total", "cumulative_energy_MWh", "load_cycles"}
base_features = [c for c in df.select_dtypes(include="number").columns if c not in excluded]
feature_groups = {"original": base_features, "snapshot": base_features + snapshot_features, "snapshot_plus_temporal": base_features + snapshot_features + temporal_features}
train_part = engineered.loc[engineered["split"] == "train"]
validation_part = engineered.loc[engineered["split"] == "validation"]
ablation_rows = []
for group_name, columns in feature_groups.items():
    diagnostic = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1))])
    diagnostic.fit(train_part[columns], train_part["failure"])
    probabilities = diagnostic.predict_proba(validation_part[columns])[:, 1]
    predictions = (probabilities >= 0.5).astype("int8")
    ablation_rows.append({"feature_group": group_name, "feature_count": len(columns), "recall_at_0_5": recall_score(validation_part["failure"], predictions), "precision_at_0_5": precision_score(validation_part["failure"], predictions, zero_division=0), "f1_at_0_5": f1_score(validation_part["failure"], predictions), "pr_auc": average_precision_score(validation_part["failure"], probabilities)})
ablation_report = pd.DataFrame(ablation_rows).set_index("feature_group").sort_values(["recall_at_0_5", "pr_auc"], ascending=False)
display(ablation_report)
selected_group = ablation_report.index[0]
selected_features = feature_groups[selected_group]
print("Selected feature group by validation recall, then PR-AUC:", selected_group)
print("Test set scored during feature selection:", False)

,feature_count,recall_at_0_5,precision_at_0_5,f1_at_0_5,pr_auc
feature_group,,,,,
snapshot_plus_temporal,86,0.6088,0.7428,0.6691,0.7993
snapshot,50,0.5158,0.7599,0.6145,0.7622
original,26,0.5138,0.7415,0.6070,0.6934


Selected feature group by validation recall, then PR-AUC: snapshot_plus_temporal
Test set scored during feature selection: False


## 6. Save Selected Unscaled Features

The output retains identifiers for traceability. Later model pipelines must explicitly select `selected_features` and fit imputation, transformation, scaling, and encoding within training or tuning folds.

In [7]:
output_columns = ["source_index", "timestamp", "turbine_id"] + selected_features + ["failure"]
saved = {}
for name in ["train", "validation", "test"]:
    output = engineered.loc[engineered["split"] == name, output_columns].sort_values(["timestamp", "turbine_id"]).reset_index(drop=True)
    path = processed_dir / f"04_{name}_engineered.csv"
    output.to_csv(path, index=False)
    reloaded = pd.read_csv(path)
    saved[name] = {"path": str(path.relative_to(project_root)), "rows": len(reloaded), "columns": reloaded.shape[1], "failure_count": int(reloaded["failure"].sum()), "duplicate_source_index": int(reloaded["source_index"].duplicated().sum())}
results_dir = project_root / "reports" / "model_results"
results_dir.mkdir(parents=True, exist_ok=True)
ablation_report.reset_index().to_csv(results_dir / "04_feature_ablation.csv", index=False)
pd.Series(selected_features, name="feature").to_csv(results_dir / "04_selected_features.csv", index=False)
save_report = pd.DataFrame(saved).T
display(save_report)
print("All exported source indices unique within split:", save_report["duplicate_source_index"].eq(0).all())

,path,rows,columns,failure_count,duplicate_source_index
train,data/processed/04_train_engineered.csv,84336,90,1877,0
validation,data/processed/04_validation_engineered.csv,21072,90,1485,0
test,data/processed/04_test_engineered.csv,26352,90,591,0


All exported source indices unique within split: True


## Key Findings Recap

- Exported 84,336 training, 21,072 validation and 26,352 test rows on the corrected split.
- Three complete test turbines are absent from development data.
- Created 24 snapshot and 36 past-only per-turbine temporal features.
- Validation selection retained snapshot_plus_temporal with 86 predictors.
- The selected feature group achieved validation recall 0.6088 and PR-AUC 0.7993.
- Test scores were not used to choose the feature group.